# 00. データ準備

このノートブックは、ハンズオンで使うカタログ・スキーマ・ボリュームを作り、PDF をボリュームに置く。後続のノートブック（1-1、1-2、2）はすべて、ここで作るボリュームを使う。

## このノートブックでやること
1. パラメータの設定（widget）
2. カタログ・スキーマ・ボリュームの作成
3. PDF をボリュームに置く
4. 置いたファイルの確認

## 前提
- サーバーレスのノートブック（環境バージョン 4 で動作確認）
- スキーマを作る権限（カタログの `USE CATALOG` と `CREATE SCHEMA`）。カタログも作る場合はメタストアの `CREATE CATALOG`
- ボリュームは **マネージドボリューム** で作る。Managed パターンの content search はマネージドボリュームだけに対応するため
- AI Search のエンドポイントは参加者全員で1つを共有する。代表者が **00-0_代表者_共通エンドポイントの作成** を先に実行しておく（参加者は実行しない）

---

## 0-1. パラメータの設定

ノートブック上部の widget で値を変えられる。まずこのセルを実行し、widget を確かめてから次へ進む。

| widget | 内容 |
|---|---|
| `CATALOG_NAME` | 使うカタログ。既存のカタログを使うのが基本 |
| `CREATE_CATALOG` | `true` にするとカタログも作る（`CREATE CATALOG` 権限が必要） |
| `SCHEMA_NAME` | ハンズオン用に作るスキーマ。既定は `rag_handson_<自分のユーザー名>` で、参加者ごとに別のスキーマになる。同じカタログを全員で共有しても名前がぶつからない |
| `VOLUME_NAME` | PDF を置くマネージドボリューム |
| `PDF_URL` | ダウンロードする PDF の URL。空欄にするとダウンロードせず、手動でアップロードしたファイルを使う |
| `PDF_FILE_NAME` | ボリュームに保存するときのファイル名 |

サンプルの PDF は、VDA QMC が公開している「Automotive SPICE PAM 4.0 日本語版」（166ページ）である。リポジトリには含めず、公式の URL からダウンロードする。別の PDF を使うときは `PDF_URL` と `PDF_FILE_NAME` を変える。

In [0]:
# パラメータ設定（widget から取得）
dbutils.widgets.text("CATALOG_NAME", "main", "カタログ名")
import re
_user = spark.sql("SELECT current_user()").first()[0]
DEFAULT_SCHEMA = "rag_handson_" + re.sub(r"[^a-z0-9]+", "_", _user.split("@")[0].lower()).strip("_")
dbutils.widgets.text("SCHEMA_NAME", DEFAULT_SCHEMA, "スキーマ名（参加者ごと）")
dbutils.widgets.text("VOLUME_NAME", "docs", "ボリューム名")
dbutils.widgets.dropdown("CREATE_CATALOG", "false", ["false", "true"], "カタログも作る")
dbutils.widgets.text("PDF_URL", "https://vda-qmc.de/wp-content/uploads/2024/02/Automotive-SPICE-PAM_40_Japanese-1.pdf", "PDF の URL（空欄なら手動アップロード）")
dbutils.widgets.text("PDF_FILE_NAME", "Automotive-SPICE-PAM_40_Japanese.pdf", "保存するファイル名")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
VOLUME_NAME = dbutils.widgets.get("VOLUME_NAME")
CREATE_CATALOG = dbutils.widgets.get("CREATE_CATALOG") == "true"
PDF_URL = dbutils.widgets.get("PDF_URL").strip()
PDF_FILE_NAME = dbutils.widgets.get("PDF_FILE_NAME").strip()

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"

print(f"カタログ: {CATALOG_NAME}（作成する: {CREATE_CATALOG}）")
print(f"スキーマ: {SCHEMA_NAME}")
print(f"ボリューム: {VOLUME_PATH}")
print(f"PDF: {PDF_URL or '（手動アップロード）'}")

## 0-2. カタログ・スキーマ・ボリュームの作成

`IF NOT EXISTS` を付けているため、何度実行しても既存のものは消えない。名前にハイフンなどが入っていても動くよう、識別子はバッククォートで囲む。

In [0]:
if CREATE_CATALOG:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOG_NAME}`")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.`{SCHEMA_NAME}` COMMENT 'RAG エージェント ハンズオン'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`{VOLUME_NAME}` COMMENT 'ハンズオンで使う文書'")

display(spark.sql(f"DESCRIBE VOLUME `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`{VOLUME_NAME}`"))

`volume_type` が `MANAGED` になっていることを確かめる。

## 0-3. PDF をボリュームに置く

`PDF_URL` が入っていれば、PDF をダウンロードしてボリュームに保存する。すでに同じ名前のファイルがあれば、ダウンロードしない。

> **外部の URL に接続できない環境の場合**：`PDF_URL` を空欄にし、手元にダウンロードした PDF を手動でアップロードする。左サイドバーの「カタログ」でボリュームを開き、「このボリュームにアップロード」からファイルを選ぶ。PDF 以外の形式（DOCX、PPTX など）も置ける。

In [0]:
import os
import requests

target = f"{VOLUME_PATH}/{PDF_FILE_NAME}"

if not PDF_URL:
    print("PDF_URL が空欄のため、ダウンロードしない。ボリュームに手動でアップロードしたファイルを使う。")
elif os.path.exists(target):
    print(f"既に存在するため、ダウンロードしない: {target}")
else:
    resp = requests.get(PDF_URL, timeout=120)
    resp.raise_for_status()
    with open(target, "wb") as f:
        f.write(resp.content)
    print(f"保存した: {target}（{len(resp.content) / 1024 / 1024:.1f} MB）")

## 0-4. 置いたファイルの確認

In [0]:
files = dbutils.fs.ls(VOLUME_PATH)
display(spark.createDataFrame([(f.name, round(f.size / 1024 / 1024, 2)) for f in files], "name STRING, size_mb DOUBLE"))
assert files, "ボリュームにファイルがない。0-3 を実行するか、手動でアップロードする。"

## まとめ

- ハンズオン用のスキーマとマネージドボリュームを作り、PDF を置いた
- 次は **1-1_Scratch_パースとAI Search索引の作成** に進む。Managed パターンだけを試す場合は **2_Managed_Genie AgentとContent Search** に進んでよい

### 後片付け
ハンズオンの最後に、README の「後片付け」の順で資源を消す。スキーマは中身を確認してから `DROP SCHEMA ... CASCADE` で消す。